# Tools
Model can request to call tools that performs such as fetching fata from a database, searching the web, or running code. Tools are paring of:
1. A schema, including the name of the tool, a discription, and argument definition
2. A function or continue to execute.

In [1]:
from langchain.chat_models import init_chat_model

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

llm = init_chat_model(
    model="llama-3.3-70b-versatile",
    model_provider="groq"
)

In [3]:
response = llm.invoke("why do parrots talk?")
response.content

"Parrots are known for their impressive ability to mimic human speech and other sounds, but why do they do it? The reasons are rooted in their evolution, behavior, and social interactions.\n\n1. **Communication**: In the wild, parrots use vocalizations to communicate with each other. They have a wide range of calls, chirps, and whistles to convey information about food, predators, and social interactions. By mimicking human speech, parrots may be using a similar form of communication to interact with their human caregivers.\n2. **Social bonding**: Parrots are highly social animals that thrive on interaction and attention. By mimicking human speech, they may be trying to form a bond with their owners or other parrots. They learn to associate certain words or phrases with rewards, such as food or affection.\n3. **Learning and problem-solving**: Parrots are highly intelligent birds, and mimicking human speech may be a way for them to learn and problem-solve. By repeating words and phrases

In [4]:
# tools
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools=llm.bind_tools([get_weather])


In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'cgxsywme9', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 220, 'total_tokens': 234, 'completion_time': 0.045790096, 'completion_tokens_details': None, 'prompt_time': 0.020941827, 'prompt_tokens_details': None, 'queue_time': 0.162693031, 'total_time': 0.066731923}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f886c-7db0-7830-aee3-62adbc340909-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'cgxsywme9', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 220, 'output_tokens': 14, 'total_tokens': 234}
Tool: get_weather
Args: {'location': 'Boston'}


In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

I can provide you with general information about the weather in Boston. Boston has a humid continental climate with cold winters and warm summers. The weather can vary significantly depending on the time of year. If you're looking for the current weather, I recommend checking a weather forecast website or app for the most up-to-date information.


In [7]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8vq2f7npq', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 219, 'total_tokens': 233, 'completion_time': 0.053661749, 'completion_tokens_details': None, 'prompt_time': 0.022137671, 'prompt_tokens_details': None, 'queue_time': 0.162856074, 'total_time': 0.07579942}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f886c-7ec4-7bc2-8bf7-d1525e433b3f-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '8vq2f7npq', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 219, 'output_tokens': 14, 'total_tokens': 233}),
 ToolMessage(content="It's 